# 🕵️ Dia 72 — Phishing e textos de golpe escritos por IA

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 72* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-72.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 27/out. A ouvidoria juntou 12 mensagens: os SMS do dia 5, WhatsApps do perfil "Serra Azul Atendimento" e quatro comunicados verdadeiros da cooperativa. A M04 começa com *"Claro! Aqui está uma versão mais formal da mensagem"*.

Você: *"O Pescador usa ChatGPT! Isso prova que é golpe."* Rafa: *"O e-mail do aplicativo novo, que a própria cooperativa mandou, também parece ChatGPT. É golpe?"*

Neste notebook você vai:

1. **Carregar as 12 mensagens** da Operação Isca.
2. **Marcar sinais de estilo e mecanismos** com expressões regulares, e ver que estilo não decide nada.
3. **Aplicar a regra do veredito** (só `mech:` decide).
4. **Achar o mesmo modelo de texto** em duas cooperativas (vínculo médio do Dia 68).
5. **Jogar o quiz "mecanismo ou estilo?"** e exportar o `gold.csv` do Dia 73.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo aqui é fictício (Operação Isca).

## 1. As mensagens

Tentamos baixar o CSV do repositório no GitHub. Sem rede, usamos a amostra embutida (idêntica). Tudo fictício: domínios `.example`, telefones `+55 00 00000-00NN`.

In [ ]:
import csv            # lê o CSV do repositório
import io             # transforma o texto baixado num "arquivo" em memória
import urllib.request # baixa o arquivo (só se houver rede)

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-72/mensagens.csv")

# Plano B offline: (id, data BRT, canal, lote, texto)
AMOSTRA = [
    ('M01', '2026-10-05 07:12', 'SMS', 'SMS 5/out', 'Serra Azul: sua chave Pix foi bloqueada por segurança. Regularize em serraazul-pix.example'),
    ('M02', '2026-10-05 07:31', 'SMS', 'SMS 5/out', 'Serra Azul: sua chave Pix foi bloqueada por segurança. Regularize em lnk.example/sa7'),
    ('M03', '2026-10-05 08:40', 'SMS', 'SMS 5/out', 'SERRA AZUL INFORMA: acesso incomum na sua conta. Evite o bloqueio total ainda hoje em serraazul-seguranca.example'),
    ('M04', '2026-10-21 19:20', 'WhatsApp', 'WhatsApp 21-25/out', 'Claro! Aqui está uma versão mais formal da mensagem: Prezado(a) cooperado(a), identificamos uma movimentação atípica em sua conta. Sua segurança é nossa prioridade, nosso compromisso e nossa missão. Um de nossos gerentes entrará em contato ainda hoje para regularizar sua chave Pix. Por segurança, não comente este atendimento com terceiros.'),
    ('M05', '2026-10-21 19:22', 'WhatsApp', 'WhatsApp 21-25/out', 'Prezado(a) [Nome do Cooperado], identificamos uma movimentação atípica em sua conta. Sua segurança é nossa prioridade, nosso compromisso e nossa missão. Um de nossos gerentes entrará em contato ainda hoje para regularizar sua chave Pix.'),
    ('M06', '2026-10-22 09:02', 'WhatsApp', 'WhatsApp 21-25/out', 'Claro! Aqui está uma versão mais formal da mensagem: Prezado(a) associado(a), identificamos uma movimentação atípica em sua conta. Sua segurança é nossa prioridade, nosso compromisso e nossa missão. Acesse coopvale-pix.example para regularizar.'),
    ('M07', '2026-10-23 10:15', 'SMS', 'WhatsApp 21-25/out', 'Serra Azul: Pix de R$ 1.890,00 agendado para hoje. Não reconhece? Ligue agora para +55 00 00000-0022 e cancele.'),
    ('M08', '2026-10-25 21:40', 'WhatsApp', 'WhatsApp 21-25/out', 'Oi seu Antônio, aqui é o Cláudio da agência Centro, salva esse meu número novo. Amanhã cedo te ligo sobre o Pix bloqueado.'),
    ('M09', '2026-09-28 10:00', 'E-mail', 'legítimas', 'Prezado cooperado, é com grande alegria que anunciamos uma nova etapa em nossa jornada: o novo aplicativo Serra Azul, mais rápido, mais seguro e mais completo, que vai transformar sua experiência. Atualize pela loja oficial do seu celular.'),
    ('M10', '2026-10-05 12:30', 'SMS', 'legítimas', 'Serra Azul: circula SMS falso sobre Pix bloqueado. Não pedimos senha nem códigos. Dúvidas: ligue para o número do verso do seu cartão.'),
    ('M11', '2026-10-02 08:00', 'E-mail', 'legítimas', 'Convocação: Assembleia Geral Ordinária em 30/10, às 19h, na sede. As inscrições para uso da palavra encerram na sexta-feira.'),
    ('M12', '2026-10-03 10:03', 'E-mail', 'legítimas', 'Você pediu para redefinir a senha do internet banking às 10:02. Se não foi você, ignore este e-mail; nenhuma alteração será feita.'),
]

try:
    with urllib.request.urlopen(URL, timeout=10) as r:          # timeout curto: não trava sem rede
        leitor = csv.DictReader(io.StringIO(r.read().decode("utf-8")))
        MSGS = [(l["id"], l["data_brt"], l["canal"], l["lote"], l["text"]) for l in leitor]
    print("✅ Baixado do repositório.")
except Exception as erro:                                          # sem rede, 404, proxy...
    MSGS = AMOSTRA
    print(f"⚠️ Sem acesso ({type(erro).__name__}); usando a amostra embutida.")

for mid, data, canal, lote, texto in MSGS:
    print(f"{mid} [{canal:8s}] {texto[:85]}{'...' if len(texto) > 85 else ''}")

## 2. Dois marcadores: estilo e mecanismo

Os rótulos são os mesmos do XML do Label Studio do original (`style:` e `mech:`), com dois mecanismos ➕ brasileiros. Cada marcador é uma expressão regular simples: **ele acha candidatos, não decide nada**. No Label Studio, quem decide o trecho é você.

Repare no comentário de cada regra: ele diz *por que* aquele padrão é suspeito.

In [ ]:
import re

# Sinais de ESTILO (style:) — fracos, nunca decidem o veredito.
ESTILO = {
    "style:chatbot_residue": r"claro! aqui est|espero que (isso|isto) ajude|\[[^\]]*(nome|link|agência)[^\]]*\]",  # sobras do chatbot
    "style:generic_greeting": r"prezad[oa]\(a\)|cooperad[oa]\(a\)|associad[oa]\(a\)|prezado cooperado",          # saudação sem nome
    "style:triad": r"\b\w+, (nosso |nossa |mais )?\w+ e (nosso |nossa |mais )?\w+\b",                             # "X, Y e Z"
    "style:inflated_or_sales": r"nova etapa|jornada|transformar|exclusiv|grande alegria",                           # importância inflada
}

# MECANISMOS (mech:) — são eles que decidem o veredito.
MECANISMO = {
    "mech:link_or_attachment": r"\b[\w-]+\.example(/\w+)?\b",                         # link para domínio (aqui, .example)
    "mech:urgency_deadline": r"ainda hoje|agora|bloqueio total|amanhã cedo",            # pressa para agir
    "mech:impersonated_sender": r"aqui é o cláudio|salva esse meu número novo",  # identidade que não bate
    "mech:off_platform_move": r"ligue agora para \+55|não comente",                     # tirar do canal oficial / isolar
    "mech:fake_block_or_regularize": r"bloquead|regulariz|movimentação atípica|acesso incomum",  # ➕ falsa central
}
# Nota honesta: o nome do perfil falso ("Serra Azul Atendimento") está no REMETENTE, não no texto.
# No Label Studio você também rotula o remetente; esta regex olha só o texto, então perde esse caso.

def marcar(texto, regras):
    """Devolve {rótulo: trecho} para cada regra que casou no texto."""
    achados = {}
    for rotulo, padrao in regras.items():
        m = re.search(padrao, texto, flags=re.IGNORECASE)
        if m:
            achados[rotulo] = m.group(0)          # o trecho exato: é o que você citaria no relatório
    return achados

print(f"{'id':4s} {'estilo':>6s} {'mech':>5s}  trechos de estilo")
TABELA = []
for mid, data, canal, lote, texto in MSGS:
    est, mec = marcar(texto, ESTILO), marcar(texto, MECANISMO)
    TABELA.append((mid, lote, est, mec, texto))
    print(f"{mid:4s} {len(est):6d} {len(mec):5d}  {list(est)}")

> 💡 Olhe a linha do **M09** (e-mail legítimo do aplicativo novo) e compare com a do **M01** (SMS do golpe). É o mesmo resultado que o original manda você observar nas suas quatro mensagens legítimas: estilo não separa golpe de comunicado.

## 3. A regra do veredito (fluxograma do Dia 72)

- Pelo menos um `mech:` → **phishing**, apoiado nos trechos de mecanismo.
- Nenhum `mech:` → **legítimo ou incerto**, tenha ou não estilo.

O marcador automático erra (veja o M10, aviso legítimo que fala de "Pix bloqueado"). Por isso a coluna final compara com o seu gabarito humano.

In [ ]:
GABARITO = {"M01": "phishing", "M02": "phishing", "M03": "phishing", "M04": "phishing",
            "M05": "phishing", "M06": "phishing", "M07": "phishing", "M08": "phishing",
            "M09": "legitimo", "M10": "legitimo", "M11": "legitimo", "M12": "legitimo"}

def veredito(mec):
    """A regra do original: só mecanismo decide."""
    return "phishing" if mec else "legitimo/incerto"

divergencias = []
for mid, lote, est, mec, texto in TABELA:
    v = veredito(mec)
    ok = v.startswith(GABARITO.get(mid, "?")[:8]) or (v != "phishing" and GABARITO.get(mid) == "legitimo")
    if not ok:
        divergencias.append(mid)
    print(f"{mid}: {v:16s} gabarito={GABARITO.get(mid):9s} {'✅' if ok else '⚠️ revisar'}  mech={list(mec)}")

print("\nRevisar à mão:", divergencias or "nenhuma")
print("Lição: o marcador acha CANDIDATOS. Um aviso legítimo que fala de 'bloqueio' engana a regex;")
print("no Label Studio você não marcaria 'mech:' ali, porque não há pedido de ação por link nem por ligação desconhecida.")

## 4. Tabela-resumo (versão Operação Isca)

O original compara antes/depois de dezembro de 2022. Nossas mensagens são todas de 2026; então comparamos **campanha** × **comunicados legítimos**: quantas trazem pelo menos um `style:`?

In [ ]:
grupos = {"campanha (phishing)": [], "comunicados legítimos": []}
for mid, lote, est, mec, texto in TABELA:
    chave = "comunicados legítimos" if GABARITO[mid] == "legitimo" else "campanha (phishing)"
    grupos[chave].append(len(est))

for nome, contagens in grupos.items():
    com_estilo = sum(1 for c in contagens if c > 0)
    print(f"{nome:24s} {com_estilo}/{len(contagens)} com pelo menos um style:  (contagens: {contagens})")
print("\nSe os legítimos pontuam estilo tanto quanto a campanha, estilo não decide caso nenhum.")

## 5. O mesmo modelo de texto em duas cooperativas

Agora a resposta à pergunta do Rafa. Quebramos cada mensagem em *shingles* de 5 palavras e medimos o índice de Jaccard entre pares (a Inovação OSINT do dia). Pares de **cooperativas diferentes** com Jaccard alto viram uma aresta **média** no grafo do Dia 68.

In [ ]:
from itertools import combinations   # todos os pares, sem repetir

def shingles(texto, n=5):
    p = re.findall(r"\w+", texto.lower())               # só palavras
    return {" ".join(p[i:i + n]) for i in range(len(p) - n + 1)}

def jaccard(a, b):
    A, B = shingles(a), shingles(b)
    return len(A & B) / len(A | B) if (A | B) else 0.0

def cooperativa(texto):
    """Qual cooperativa a mensagem imita (pelo texto)."""
    return "Coopvale" if "coopvale" in texto.lower() else "Serra Azul"

pares = []
for (i1, *_, t1), (i2, *_, t2) in combinations(MSGS, 2):
    j = jaccard(t1, t2)
    if j >= 0.3:                                        # limiar didático; registre o que usou
        pares.append((j, i1, i2, cooperativa(t1), cooperativa(t2)))

for j, a, b, ca, cb in sorted(pares, reverse=True):
    cruzado = " ← cooperativas diferentes!" if ca != cb else ""
    print(f"{a}–{b}  Jaccard={j:.2f}  ({ca} / {cb}){cruzado}")

ARESTAS = [f"{a},{b},Undirected,mesmo modelo de texto (Jaccard {j:.2f}),2"
           for j, a, b, ca, cb in pares if ca != cb]
print("\nArestas para o grafo (peso 2 = médio):")
print("Source,Target,Type,Label,Weight")
print("\n".join(ARESTAS) or "(nenhuma)")

> 🧠 **Por que médio e não forte?** Porque o modelo de texto (ou o próprio kit) pode ter sido vendido para vários grupos. Só uma chave Pix, conta, telefone ou domínio em comum faz a ligação forte. O "Claro! Aqui está" prova **como** a mensagem foi escrita e serve para **ligar casos**; quem decide se é golpe são os mecanismos.

## 6. 🎮 Quiz: mecanismo ou estilo?

Para cada trecho, responda **m** (mecanismo, decide o veredito) ou **e** (estilo, só apoio/agrupamento). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("🇺🇸 'within 24 hours'", "m", "Urgência/prazo é mecanismo."),
    ("🇺🇸 'faster, safer, and more rewarding'", "e", "Tríade forçada: estilo."),
    ("🇺🇸 'I hope this helps!'", "e", "Resíduo de chatbot: estilo (o mais forte deles, mas estilo)."),
    ("🇺🇸 're-verify your direct deposit details ... here: [link]'", "m", "Pedido de credencial + link."),
    ("🇧🇷 'Claro! Aqui está uma versão mais formal da mensagem'", "e", "Resíduo: diz como foi escrito, não se é golpe."),
    ("🇧🇷 'Regularize em serraazul-pix.example'", "m", "Falsa regularização + link para domínio imitação."),
    ("🇧🇷 'prioridade, compromisso e missão'", "e", "Tríade forçada."),
    ("🇧🇷 'salva esse meu número novo'", "m", "Remetente que se passa por outro (identidade que não bate)."),
    ("🇧🇷 'Por segurança, não comente este atendimento com terceiros'", "m", "Sigilo/isolamento: mecanismo (BEC chama de secrecy)."),
    ("🇧🇷 'Prezado(a) [Nome do Cooperado]'", "e", "Saudação genérica + marcador não preenchido."),
]

acertos = 0
for i, (trecho, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {trecho}")
    if MODO_JOGO:
        r = ""
        for _ in range(5):                               # no máximo 5 tentativas: nunca trava
            r = input("   m (mecanismo) ou e (estilo)? ").strip().lower()[:1]
            if r in ("m", "e"):
                break
        acertos += (r == certo)
        print("   ✅" if r == certo else f"   ❌ era {certo}")
    print("   →", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 7. 💾 Exportar o `gold.csv` do Dia 73

Mesmo formato do original (`id`, `text`, `gold_verdict` com `phishing`/`legitimate`/`unsure`). O Dia 73 acrescenta as duas linhas adversariais.

In [ ]:
from pathlib import Path

Path("notas").mkdir(exist_ok=True)
mapa = {"phishing": "phishing", "legitimo": "legitimate"}
with open("notas/gold.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "text", "gold_verdict"])
    for mid, data, canal, lote, texto in MSGS:
        w.writerow([mid, texto, mapa.get(GABARITO.get(mid), "unsure")])
print("💾 notas/gold.csv —", len(MSGS), "linhas")

try:
    from google.colab import files
    files.download("notas/gold.csv")
except ImportError:
    print("(Fora do Colab: ficou em notas/)")

## ✅ Antes de fechar o Dia 72

- [ ] 20 mensagens publicadas rotuladas no Label Studio + tabela antes/depois de dez/2022
- [ ] 12 mensagens da Operação Isca rotuladas; todo phishing com pelo menos um `mech:`
- [ ] Aresta "mesmo modelo de texto" (peso médio) proposta para o grafo
- [ ] `notas/gold.csv` salvo para amanhã

**Amanhã (Dia 73):** você entrega a taxonomia a um modelo local e mede onde ele erra. Rafa já avisou: *"uma das mensagens vai tentar mandar no seu modelo."*

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>